<a href="https://colab.research.google.com/github/xmarx11/mk8dx-proyecto/blob/main/mk8dx_proyecto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Desempeño de combinaciones en Mario Kart 8 Deluxe

**Pregunta:** ¿Los atributos de los componentes (personaje, carrocería, llantas y planeador) afectan el desempeño del jugador?

Este notebook sigue las secciones 2 a 4 del documento del proyecto:

1. Carga de las fuentes de datos
2. Supervisión de los datos: estructura, duplicados, faltantes y consistencia entre fuentes
3. Procesamiento: limpieza, conversiones de tipo y variables nuevas
4. Dataset final listo para aprendizaje
5. Muestra aleatoria de 100 registros en CSV

**Fuentes**

- Récords de contrarreloj: [mkwrs.com/data/mk8dx_wrs.csv](https://mkwrs.com/data/mk8dx_wrs.csv)
- Estadísticas por pieza: [Kaggle, *Mario Kart 8 Deluxe Stats* (tuvaludelsman)](https://www.kaggle.com/datasets/tuvaludelsman/mario-kart-8-deluxe-stats)
- Validación: [Kaggle, *Mario Kart 8 Deluxe All Combo Stats* (nicksobota)](https://www.kaggle.com/datasets/nicksobota/mario-kart-8-deluxe-all-combo-stats)

In [54]:
# Descarga los datos desde el repositorio de GitHub del proyecto
import os, urllib.request, urllib.parse, zipfile

BASE = "https://raw.githubusercontent.com/xmarx11/mk8dx-proyecto/main/"

def descargar(ruta):
    destino = os.path.dirname(ruta)
    if destino:
        os.makedirs(destino, exist_ok=True)
    urllib.request.urlretrieve(BASE + urllib.parse.quote(ruta), ruta)

descargar("mk8dx_wrs.csv")
for t in ["Bodies", "Drivers", "Gliders", "Tyres"]:
    descargar(f"stats/Mario Kart 8 Deluxe Stats - Stats ({t}).csv")

descargar("archive.zip")
zipfile.ZipFile("archive.zip").extractall()

print(os.listdir("stats"))

['Mario Kart 8 Deluxe Stats - Stats (Bodies).csv', 'Mario Kart 8 Deluxe Stats - Stats (Drivers).csv', 'Mario Kart 8 Deluxe Stats - Stats (Gliders).csv', 'Mario Kart 8 Deluxe Stats - Stats (Tyres).csv']


## 0. Configuración

Archivos esperados en Colab:

- `mk8dx_wrs.csv` (si no está, se descarga de mkwrs.com)
- Carpeta `stats/` con los cuatro CSV de tuvaludelsman
- `combos.csv` de nicksobota (opcional; solo se usa para la validación de la sección 2.5)

In [55]:
import os, re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
SEED = 42

RUTA_WRS = "mk8dx_wrs.csv"
URL_WRS = "https://mkwrs.com/data/mk8dx_wrs.csv"
RUTA_STATS = "stats"
RUTA_COMBOS = "combos.csv"

# Ventana de versión del juego (sección 3, paso 7)
FECHA_INICIO = "2023-11-09"   # v3.0.0: última actualización con ajustes de estadísticas (supuesto a verificar)
FECHA_FIN    = "2026-09-02"   # v4.0.0: aumento general de velocidad; se excluye desde esta fecha

## 1. Carga de datos

### 1.1 Récords mundiales (mkwrs.com)

El CSV no trae encabezados. Se le asignan nombres según el diccionario de la sección 3.1 del documento (paso 1 del procesamiento).

In [56]:
fuente = RUTA_WRS if os.path.exists(RUTA_WRS) else URL_WRS
cols = ["id_registro","jugador","fecha","pista","tiempo_ms","id_pista","pais","video",
        "monedas_vuelta","hongos_vuelta","carroceria","llantas","flag_12","planeador",
        "dias_vigente","personaje","vuelta_1","vuelta_2","vuelta_3","flag_19","flag_20",
        "vuelta_4","vuelta_5","vuelta_6","vuelta_7","cc","nota","flag_27","vacia"]
wrs = pd.read_csv(fuente, header=None, names=cols, keep_default_na=False, na_values=[""])
print("Fuente:", fuente)
print("Dimensiones:", wrs.shape)
wrs.head(3)

Fuente: mk8dx_wrs.csv
Dimensiones: (14255, 29)


,id_registro,jugador,fecha,pista,tiempo_ms,id_pista,pais,video,monedas_vuelta,hongos_vuelta,carroceria,llantas,flag_12,planeador,dias_vigente,personaje,vuelta_1,vuelta_2,vuelta_3,flag_19,flag_20,vuelta_4,vuelta_5,vuelta_6,vuelta_7,cc,nota,flag_27,vacia
0,2588,B!KZO,2017-04-28,SNES Donut Plains 3,79969,22,Japan,N/A,7-0-0,1-1-1,Streetle,Slim,0,Plane Glider,0,Morton,27.378,26.351,26.240,1,0,NaN,NaN,NaN,NaN,150,0,0,NaN
1,2614,Mattew,2017-04-28,Thwomp Ruins,114981,4,France,N/A,10-0-0,1-1-1,Blue Falcon,Slim,0,Super Glider,0,Heavy Mii,40.114,37.563,37.304,1,0,NaN,NaN,NaN,NaN,150,0,0,NaN
2,2656,Mattew,2017-04-28,N64 Yoshi Valley,123606,28,France,N/A,-,-,-,-,0,-,0,-,-,-,-,1,0,NaN,NaN,NaN,NaN,150,0,0,NaN


### 1.2 Estadísticas por pieza (tuvaludelsman)

Se cargan sin modificar para poder supervisarlas antes de limpiarlas.

In [57]:
TIPOS = ["Bodies", "Tyres", "Gliders", "Drivers"]
STATS = ["WG","AC","ON","OF","MT","SL","SW","SA","SG","TL","TW","TA","TG","IV"]

def leer_crudo(tipo):
    ruta = os.path.join(RUTA_STATS, f"Mario Kart 8 Deluxe Stats - Stats ({tipo}).csv")
    return pd.read_csv(ruta, header=None, dtype=str)

crudos = {t: leer_crudo(t) for t in TIPOS}
crudos["Drivers"].head(4)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,Drivers (DV),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Number,Driver,NaN,WG,AC,ON,OF,MT,SL,SW,SA,SG,TL,TW,TA,TG,IV
2,#1,NaN,Mario\nMro,6,2,4,2,3,7,7,7,7,4,4,4,4,3
3,#2,NaN,Luigi\nLig,6,2,5,1,3,7,7,7,7,5,5,5,5,3


### 1.3 Combinaciones (nicksobota, opcional)

In [58]:
combos = pd.read_csv(RUTA_COMBOS, index_col=0) if os.path.exists(RUTA_COMBOS) else None
print("combos.csv cargado:", combos.shape if combos is not None else "no disponible")

combos.csv cargado: (703560, 18)


## 2. Análisis de Datos

### 2.1 Estructura y tipos de los récords

In [59]:
wrs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14255 entries, 0 to 14254
Data columns (total 29 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_registro     14255 non-null  int64  
 1   jugador         14255 non-null  object 
 2   fecha           14255 non-null  object 
 3   pista           14255 non-null  object 
 4   tiempo_ms       14255 non-null  int64  
 5   id_pista        14255 non-null  int64  
 6   pais            14255 non-null  object 
 7   video           14255 non-null  object 
 8   monedas_vuelta  14255 non-null  object 
 9   hongos_vuelta   14255 non-null  object 
 10  carroceria      14255 non-null  object 
 11  llantas         14255 non-null  object 
 12  flag_12         14255 non-null  int64  
 13  planeador       14255 non-null  object 
 14  dias_vigente    14255 non-null  int64  
 15  personaje       14255 non-null  object 
 16  vuelta_1        14255 non-null  object 
 17  vuelta_2        14255 non-null 

### 2.2 Registros duplicados

In [60]:
print("Identificadores repetidos:", wrs["id_registro"].duplicated().sum())
print("Filas idénticas (sin contar el id):", wrs.drop(columns="id_registro").duplicated().sum())
print("Mismo jugador, fecha, pista, tiempo y cc:",
      wrs.duplicated(subset=["jugador","fecha","pista","tiempo_ms","cc"]).sum())

Identificadores repetidos: 0
Filas idénticas (sin contar el id): 0
Mismo jugador, fecha, pista, tiempo y cc: 0


### 2.3 Datos faltantes

mkwrs marca los datos faltantes con `-`, no con celdas vacías, así que se cuentan por separado.

In [61]:
piezas_cols = ["carroceria","llantas","planeador","personaje"]
sin_combo = (wrs[piezas_cols] == "-").any(axis=1)
sin_vueltas = (wrs[["vuelta_1","vuelta_2","vuelta_3"]] == "-").any(axis=1)
sin_video = wrs["video"].isna() | (wrs["video"] == "N/A")
print(f"Sin combinación registrada: {sin_combo.sum()} ({sin_combo.mean():.1%})")
print(f"Sin tiempos por vuelta: {sin_vueltas.sum()} (con combinación: {(sin_vueltas & ~sin_combo).sum()})")
print(f"Sin video: {sin_video.sum()}")

Sin combinación registrada: 248 (1.7%)
Sin tiempos por vuelta: 286 (con combinación: 52)
Sin video: 6230


### 2.4 Tablas de estadísticas por pieza

In [62]:
for t, d in crudos.items():
    datos = d.iloc[2:]                                  # filas debajo del título y el encabezado
    datos = datos[datos[0].astype(str).str.startswith("#")]   # solo filas de piezas (sin notas al pie)
    nombres = datos[2].str.split("\n").str[0].str.strip()
    simbolos = datos.iloc[:, 3:].apply(lambda s: s.str.contains("🟊", na=False)).sum().sum()
    print(f"{t:8s} piezas: {len(datos):3d} | nombres repetidos: {nombres.duplicated().sum()} "
          f"| filas con estadísticas idénticas: {datos.iloc[:, 3:].duplicated().sum():2d} | valores con 🟊: {simbolos}")

Bodies   piezas:  41 | nombres repetidos: 0 | filas con estadísticas idénticas: 14 | valores con 🟊: 63
Tyres    piezas:  22 | nombres repetidos: 0 | filas con estadísticas idénticas:  8 | valores con 🟊: 54
Gliders  piezas:  15 | nombres repetidos: 0 | filas con estadísticas idénticas: 11 | valores con 🟊: 102
Drivers  piezas:  52 | nombres repetidos: 0 | filas con estadísticas idénticas: 35 | valores con 🟊: 65


Las filas con estadísticas idénticas no son errores: el juego agrupa piezas en clases que comparten estadísticas (por ejemplo, Mario y Ludwig).

Las notas al pie de la tabla de personajes explican el caso de los Mii:

In [63]:
notas = crudos["Drivers"][~crudos["Drivers"][0].astype(str).str.startswith(("#", "Number", "Drivers"))]
for n in notas[0].dropna():
    print(n)

1: Even though there are multiple versions of the characters having different colors, only one set of statistics is defined for them.
2: Small-sized Miis are programmed to have the same statistics as Baby Mario.
3: Large-sized Miis are programmed to have the same statistics as Dry Bowser.
All data sourced from https://www.mariowiki.com/Mario_Kart_8_Deluxe_in-game_statistics


### 2.5 Combinaciones y consistencia entre fuentes de Kaggle

La función `limpiar_piezas` deja cada tabla de Kaggle con un nombre de pieza y sus 14 estadísticas como números (quita la fila de título, las notas al pie y el símbolo 🟊).

Después se comprueba cómo se calculan las estadísticas de una combinación: se toman 2,000 combinaciones al azar de `combos.csv`, se suman las estadísticas de sus cuatro piezas y se comparan. Si coinciden al 100%, basta con sumar las piezas y no es necesario cargar `combos.csv` (88 MB) para construir el dataset.

In [64]:
def limpiar_piezas(d, tipo):
    """Paso 2 del procesamiento: deja una tabla nombre -> 14 estadísticas numéricas."""
    d = d.iloc[1:].copy()                                # quita la fila de título
    d.columns = d.iloc[0]; d = d.iloc[1:]                # usa el encabezado real
    d = d[d["Number"].astype(str).str.startswith("#")]   # quita notas al pie
    d["nombre"] = (d.iloc[:, 2].str.split("\n").str[0]
                   .str.replace(r"[¹²³]", "", regex=True).str.strip())
    for s in STATS:
        d[s] = pd.to_numeric(d[s].str.replace("🟊", "", regex=False))
    return d.set_index("nombre")[STATS]

piezas = {t: limpiar_piezas(d, t) for t, d in crudos.items()}

if combos is not None:
    print("Combinaciones repetidas:", combos.duplicated(subset=["Body","Tire","Glider","Driver"]).sum())
    print("Valores faltantes:", combos.isna().sum().sum())
    print("Rango de estadísticas:", combos[STATS].min().min(), "a", combos[STATS].max().max())

    base = lambda s: re.sub(r"[¹²³]", "", s.split("/")[0]).strip()
    m = combos.sample(2000, random_state=SEED)
    suma = sum(piezas[t].loc[m[c].map(base)].to_numpy()
               for t, c in [("Bodies","Body"),("Tyres","Tire"),("Gliders","Glider"),("Drivers","Driver")])
    iguales = (suma == m[STATS].to_numpy()).all(axis=1)
    print(f"Combinaciones donde la suma de piezas coincide con nicksobota: {iguales.mean():.0%} de {len(m)}")
else:
    print("combos.csv no disponible: se omite la validación (resultado documentado: 100% de coincidencia).")

Combinaciones repetidas: 0
Valores faltantes: 0
Rango de estadísticas: 0.0 a 20.0
Combinaciones donde la suma de piezas coincide con nicksobota: 100% de 2000


### 2.6 Compatibilidad de nombres entre fuentes

Piezas de los récords que no aparecen tal cual en las tablas de Kaggle, antes de normalizar:

In [65]:
mapa = {"carroceria":"Bodies", "llantas":"Tyres", "planeador":"Gliders", "personaje":"Drivers"}
con_combo = wrs[~sin_combo]
for col, tipo in mapa.items():
    faltan = con_combo.loc[~con_combo[col].isin(piezas[tipo].index), col].value_counts()
    print(f"{col}: {faltan.sum()} registros, {len(faltan)} nombres distintos. Ejemplos: {faltan.head(6).to_dict()}")

carroceria: 0 registros, 0 nombres distintos. Ejemplos: {}
llantas: 0 registros, 0 nombres distintos. Ejemplos: {}
planeador: 1 registros, 1 nombres distintos. Ejemplos: {'Wario Glider': 1}
personaje: 3599 registros, 84 nombres distintos. Ejemplos: {'Yellow Yoshi': 488, 'Light-blue Yoshi': 432, 'Pink Yoshi': 394, 'White Yoshi': 329, 'Black Yoshi': 249, 'Heavy Mii': 244}


## 3. Procesamiento

Los pasos 1 (nombres de columnas) y 2 (limpieza de tablas de estadísticas) ya se aplicaron en las secciones 1.1 y 2.5.

### Paso 3. Eliminar récords sin combinación

In [66]:
df = wrs[~sin_combo].copy()
print("Registros:", len(wrs), "->", len(df))

Registros: 14255 -> 14007


### Paso 4. Normalizar nombres de personajes

**mkwrs** registra el personaje con su variante de color o disfraz (por ejemplo, "Yellow Yoshi" o "Heavy Mii Varia"), pero Kaggle solo tiene un nombre por personaje. Como las variantes de color no cambian las estadísticas, se agrupan en el personaje base.

Los Mii son un caso especial: sus estadísticas dependen de su tamaño. Según las notas al pie del dataset de Kaggle, el Mii pequeño tiene las mismas estadísticas que Baby Mario y el grande las mismas que Dry Bowser. El nombre original se conserva en la columna `personaje`.

In [67]:
def normalizar_personaje(n):
    n = n.strip()
    if n.startswith("Heavy Mii"):  return "Dry Bowser"     # nota 3: Mii grande = Dry Bowser
    if n.startswith("Light Mii"):  return "Baby Mario"     # nota 2: Mii pequeño = Baby Mario
    if n.startswith("Medium Mii"): return "Mii (medium)"
    if re.search(r"Yoshi$", n):    return "Yoshi"          # variantes de color
    if n.startswith("Birdo"):      return "Birdo"
    if "Inkling Girl" in n:        return "Inkling Girl"
    if "Inkling Boy" in n:         return "Inkling Boy"
    if "Shy Guy" in n:             return "Shy Guy"
    equivalencias = {"Male Villager":"Villager (male)", "Villager":"Villager (male)",
                     "Female Villager":"Villager (female)",
                     "Champion Link":"Link (Breath of the Wild)"}
    return equivalencias.get(n, n)

df["personaje_base"] = df["personaje"].map(normalizar_personaje)
print("Nombres originales:", df["personaje"].nunique(), "-> personajes base:", df["personaje_base"].nunique())

Nombres originales: 125 -> personajes base: 47


### Paso 5. Eliminar piezas sin estadísticas

In [68]:
mapa = {"carroceria":"Bodies", "llantas":"Tyres", "planeador":"Gliders", "personaje_base":"Drivers"}
validos = pd.Series(True, index=df.index)
for col, tipo in mapa.items():
    ok = df[col].isin(piezas[tipo].index)
    if (~ok).any():
        print(f"{col} sin estadísticas: {df.loc[~ok, col].value_counts().to_dict()}")
    validos &= ok
df = df[validos].copy()
print("Registros:", len(df))

planeador sin estadísticas: {'Wario Glider': 1}
Registros: 14006


### Paso 6. Unir con estadísticas

La estadística de la combinación es la suma de sus cuatro piezas (validado en la sección 2.5).

In [69]:
NOMBRES_STATS = {
    "WG":"peso", "AC":"aceleracion", "ON":"traccion_pista", "OF":"traccion_fuera",
    "MT":"mini_turbo", "SL":"vel_tierra", "SW":"vel_agua", "SA":"vel_antigrav", "SG":"vel_aire",
    "TL":"manejo_tierra", "TW":"manejo_agua", "TA":"manejo_antigrav", "TG":"manejo_aire",
    "IV":"invencibilidad"}
cols_stats = list(NOMBRES_STATS.values())

total = sum(piezas[tipo].loc[df[col]].to_numpy() for col, tipo in mapa.items())
df[cols_stats] = total
df[["personaje_base","carroceria","llantas","planeador"] + cols_stats[:5]].head()

,personaje_base,carroceria,llantas,planeador,peso,aceleracion,traccion_pista,traccion_fuera,mini_turbo
0,Morton,Streetle,Slim,Plane Glider,14,9,12,7,10
1,Dry Bowser,Blue Falcon,Slim,Super Glider,12,6,11,6,8
3,Villager (male),Standard Kart,Off-Road,Super Glider,11,11,14,11,13
5,King Boo,W 25 Silver Arrow,Button,Super Glider,9,12,14,12,13
6,Morton,Splat Buggy,Slim,Super Glider,13,6,12,5,8


### Conversiones de tipo

Se hacen antes del filtro por fecha porque el filtro necesita la columna `fecha` como fecha.

In [70]:
df["fecha"] = pd.to_datetime(df["fecha"], format="%Y-%m-%d")
df["tiempo_s"] = df["tiempo_ms"] / 1000
for v in ["vuelta_1","vuelta_2","vuelta_3"]:
    df[v] = pd.to_numeric(df[v], errors="coerce")       # "-" -> nulo
df[["fecha","tiempo_s","vuelta_1","vuelta_2","vuelta_3"] + cols_stats[:3]].dtypes

,0
fecha,datetime64[ns]
tiempo_s,float64
vuelta_1,float64
vuelta_2,float64
vuelta_3,float64
peso,int64
aceleracion,int64
traccion_pista,int64


### Paso 7. Filtrar por versión del juego

Las estadísticas de Kaggle corresponden a la versión más reciente con ajustes de balance, pero los récords van de 2017 a 2026. Un récord de 2017 se logró con estadísticas distintas, así que unirlo con las actuales sería incorrecto. Además, la versión 4.0.0 (2 de septiembre de 2026) aumentó la velocidad general, por lo que esos tiempos no son comparables.

Por eso solo se conservan los récords dentro de la ventana definida en `FECHA_INICIO` y `FECHA_FIN`.

In [71]:
df = df[(df["fecha"] >= FECHA_INICIO) & (df["fecha"] < FECHA_FIN)].copy()
print(f"Registros: {len(df)} | Pistas: {df['pista'].nunique()} | Jugadores: {df['jugador'].nunique()}")
print(df["cc"].value_counts().to_dict())

Registros: 1952 | Pistas: 95 | Jugadores: 110
{200: 996, 150: 956}


### Variables nuevas

- `dias_desde_inicio`: días entre el inicio de la ventana y el récord.
- `pct_sobre_mejor` (variable objetivo): (tiempo / mejor tiempo de la pista y categoría − 1) × 100.


`pct_sobre_mejor` es necesaria porque cada pista dura distinto: comparar 1:05 en una pista con 2:10 en otra no dice nada. Al medir qué porcentaje más lento es cada récord respecto al mejor de su pista y categoría, todos los tiempos quedan en la misma escala. Un valor de 0 es el mejor récord; un valor de 1.5 significa 1.5% más lento.

`dias_desde_inicio` sirve como control: los récords mejoran con el tiempo, y esta variable permite separar ese efecto del efecto de la combinación.

In [72]:
df["dias_desde_inicio"] = (df["fecha"] - pd.Timestamp(FECHA_INICIO)).dt.days
mejor = df.groupby(["pista","cc"])["tiempo_ms"].transform("min")
df["pct_sobre_mejor"] = (df["tiempo_ms"] / mejor - 1) * 100
df[["pista","cc","tiempo_s","pct_sobre_mejor"]].head()

,pista,cc,tiempo_s,pct_sobre_mejor
10719,Wii DK Summit,150,114.801,0.850369
10915,Tour Piranha Plant Cove,200,84.067,0.539370
10916,GBA Sunset Wilds,200,67.131,0.220952
11033,Tour Piranha Plant Cove,150,119.514,2.200255
11327,Mario Kart Stadium,200,67.154,2.658412


### Paso 8. Descartar columnas sin uso

In [73]:
descartar = ["id_registro","id_pista","video","monedas_vuelta","hongos_vuelta","dias_vigente",
             "vuelta_4","vuelta_5","vuelta_6","vuelta_7","flag_12","flag_19","flag_20","flag_27",
             "nota","vacia","tiempo_ms"]
df = df.drop(columns=descartar)
print("Columnas restantes:", df.shape[1])

Columnas restantes: 30


## 4. Dataset final

In [74]:
orden = (["fecha","dias_desde_inicio","jugador","pais","pista","cc",
          "personaje","personaje_base","carroceria","llantas","planeador"]
         + cols_stats + ["vuelta_1","vuelta_2","vuelta_3","tiempo_s","pct_sobre_mejor"])
dataset = df[orden].reset_index(drop=True)

print("Dimensiones:", dataset.shape)
print("Valores nulos:", dataset.isna().sum().sum())
print("Filas duplicadas:", dataset.duplicated().sum())
print("Personajes base:", dataset["personaje_base"].nunique(),
      "| Carrocerías:", dataset["carroceria"].nunique(),
      "| Llantas:", dataset["llantas"].nunique(),
      "| Planeadores:", dataset["planeador"].nunique())
dataset.dtypes

Dimensiones: (1952, 30)
Valores nulos: 0
Filas duplicadas: 0
Personajes base: 29 | Carrocerías: 19 | Llantas: 18 | Planeadores: 13


,0
fecha,datetime64[ns]
dias_desde_inicio,int64
jugador,object
pais,object
pista,object
cc,int64
personaje,object
personaje_base,object
carroceria,object
llantas,object


### Codificación para modelos

La categoría y la pista se transforman en variables binarias (one-hot), descartando una categoría de cada una.

In [75]:
X = pd.get_dummies(dataset[cols_stats + ["dias_desde_inicio","cc","pista"]],
                   columns=["cc","pista"], drop_first=True, dtype=int)
y = dataset["pct_sobre_mejor"]
print("X:", X.shape, "| y:", y.shape)

dataset.to_csv("mk8dx_dataset_limpio.csv", index=False)
print("Guardado mk8dx_dataset_limpio.csv")

X: (1952, 110) | y: (1952,)
Guardado mk8dx_dataset_limpio.csv


## 5. Muestra aleatoria de 100 registros

In [76]:
muestra = dataset.sample(n=100, random_state=SEED)
muestra.to_csv("mk8dx_muestra_100.csv", index=False)
print("Guardado mk8dx_muestra_100.csv:", muestra.shape)
muestra.head()

Guardado mk8dx_muestra_100.csv: (100, 30)


,fecha,dias_desde_inicio,jugador,pais,pista,cc,personaje,personaje_base,carroceria,llantas,planeador,peso,aceleracion,traccion_pista,traccion_fuera,mini_turbo,vel_tierra,vel_agua,vel_antigrav,vel_aire,manejo_tierra,manejo_agua,manejo_antigrav,manejo_aire,invencibilidad,vuelta_1,vuelta_2,vuelta_3,tiempo_s,pct_sobre_mejor
1615,2025-06-04,573,Alberto,Spain,Sunshine Airport,200,Luigi,Luigi,Teddy Buggy,Azure Roller,Paper Glider,8,15,10,9,17,9,13,11,15,14,11,13,15,4,30.204,28.730,28.754,87.688,0.000000
1406,2025-01-04,422,Pii,Norway,Ice Ice Outpost,150,White Yoshi,Yoshi,Biddybuggy,Azure Roller,Paper Glider,4,18,5,12,19,6,11,9,11,15,13,15,15,1,36.146,34.246,34.130,104.522,0.268606
1906,2026-06-28,962,kwartz,UK,N64 Rainbow Road,200,Dry Bowser,Dry Bowser,Mach 8,Leaf Tires,Paper Glider,12,10,9,9,12,14,16,18,17,7,6,10,7,11,19.122,17.707,19.466,56.295,0.008883
1580,2025-05-05,543,Zae,USA,Tour Athens Dash,150,Pink Yoshi,Yoshi,Biddybuggy,Azure Roller,Paper Glider,4,18,5,12,19,6,11,9,11,15,13,15,15,1,34.694,25.749,37.016,97.459,1.189871
307,2023-11-13,4,Alberto,Spain,Tour Madrid Drive,150,Red Yoshi,Yoshi,Mr. Scooty,Azure Roller,Paper Glider,4,18,5,12,19,6,11,9,11,15,13,15,15,1,39.283,42.558,35.088,116.929,1.084072
